# Informe de personal — hazlo tú

**Nombre:**  
**Fecha:**  

Piloto de Matplotlib. La dirección de una empresa de servicios digitales quiere revisar la política salarial y la carga de trabajo. Tu informe debe ayudarle a decidir.

- **Trabajo:** en pareja.
- **Entrega:** individual, un cuaderno por estudiante.
- **Valoración:** hecho / no hecho; no se corrige el acierto.
- **Ficha completa:** [caso-informe-personal.md](caso-informe-personal.md).

No hay que descargar datos: se generan en la celda siguiente. Son datos sintéticos y no representan personas reales.


## 0. Preparación

Ejecuta la celda una vez. Comprueba que el kernel pertenece al entorno de visualización que has elegido. Si falla un import, anota el error y revisa el entorno; no instales en el Python del sistema.

Los datos tienen 240 empleados y 12 meses. La semilla fija permite repetir el mismo resultado.


In [ ]:
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(sys.executable)
print('pandas:', pd.__version__, '| Matplotlib:', matplotlib.__version__)
plt.style.use('default')

rng = np.random.default_rng(42)
n = 240
departamentos = np.array(['Datos'] * 60 + ['Web'] * 60 + ['Sistemas'] * 60 + ['Soporte'] * 60)
base = {'Datos': 2900, 'Web': 2500, 'Sistemas': 2700, 'Soporte': 2200}

experiencia = rng.integers(0, 26, n)
salario = np.array([base[d] for d in departamentos]) + 45 * experiencia + rng.normal(0, 250, n)
idx_atipicos = rng.choice(n, 8, replace=False)
salario[idx_atipicos] *= rng.uniform(2.2, 3.2, len(idx_atipicos))

edad = np.clip(22 + experiencia + rng.integers(0, 10, n), 22, 64)
horas_semana = rng.normal(40, 3, n).round(1)
satisfaccion = rng.normal(7, 1.5, n).clip(1, 10).round(1)
satisfaccion[rng.choice(n, 12, replace=False)] = np.nan

empleados = pd.DataFrame({
    'id': np.arange(1, n + 1),
    'departamento': departamentos,
    'edad': edad,
    'experiencia': experiencia,
    'salario': salario.round(2),
    'horas_semana': horas_semana,
    'satisfaccion': satisfaccion,
})

mensual = pd.DataFrame({
    'mes': np.arange(1, 13),
    'proyectos': [9, 10, 11, 12, 13, 12, 10, 9, 12, 14, 15, 17],
    'soporte': [12, 11, 13, 12, 14, 15, 13, 12, 16, 18, 21, 23],
    'horas_extra': [6, 5, 7, 6, 9, 10, 8, 7, 12, 15, 19, 22],
})

salida = Path('salidas_estaticas')
salida.mkdir(exist_ok=True)

print('Empleados:', empleados.shape, '| ausentes en satisfaccion:', empleados['satisfaccion'].isna().sum())
print('Meses:', len(mensual))
assert empleados.shape == (240, 7)
assert empleados['satisfaccion'].isna().sum() == 12
assert len(mensual) == 12
empleados.head()

## Situación 1 — Evolución mensual

**Pregunta:** ¿cómo ha cambiado la carga de trabajo a lo largo del año?

**Tarea:** con `mensual`, crea `fig_lineas, ax_lineas = plt.subplots(...)` y dibuja `proyectos` y `soporte` frente a `mes`. Usa dos líneas con marcadores distintos, leyenda, título y ejes con unidades. No uses `plt.plot` directamente.

**Salida esperada:** 12 meses y dos series; el soporte alcanza su máximo en diciembre.

**Cuando termines, ejecuta la celda de comprobación.** Si falla, revisa los nombres de las variables y las etiquetas.


In [ ]:
# TODO: crea fig_lineas y ax_lineas; dibuja las dos series.
# Requisitos: ax_lineas.plot(...), marcadores distintos, label, leyenda, título, xlabel, ylabel.


In [ ]:
assert len(mensual) == 12
assert mensual.loc[mensual['soporte'].idxmax(), 'mes'] == 12
assert 'fig_lineas' in globals() and 'ax_lineas' in globals()
assert len(ax_lineas.lines) >= 2
assert ax_lineas.get_legend() is not None
print('Situación 1: comprobación superada')

## Situación 2 — Salario por departamento

**Pregunta:** ¿qué departamento cobra más y qué medida lo representa mejor?

**Tarea:** calcula `medias` y `medianas` del salario por departamento con `groupby`. Crea `fig_salarios, ax_salarios = plt.subplots(...)` y dibuja barras agrupadas con ambas medidas. Añade título, unidades y leyenda.

La media puede verse afectada por los salarios extremos; la mediana no.

**Salida esperada:** cuatro departamentos; Datos tiene la mediana más alta y su media supera a la mediana por los valores extremos.


In [ ]:
# TODO: calcula medias y medianas; crea fig_salarios y ax_salarios.
# ax_salarios: barras agrupadas (media y mediana) con leyenda.


In [ ]:
assert set(medias.index) == {'Datos', 'Web', 'Sistemas', 'Soporte'}
assert set(medianas.index) == set(medias.index)
assert medianas['Datos'] > medianas['Soporte']
assert len(fig_salarios.axes) == 1
assert len(ax_salarios.patches) >= 8
assert ax_salarios.get_legend() is not None
print('Situación 2: comprobación superada')

## Situación 3 — Distribución y valores extremos

**Pregunta:** ¿cómo se distribuye el salario y hay valores extremos por departamento?

**Tarea:** crea `fig_distribucion, axs_distribucion = plt.subplots(1, 2, ...)`.

- Izquierda: histograma con `bins=12` y `density=False`.
- Derecha: boxplot del salario por departamento. Construye `grupos` en el orden de `nombres` y rotula con `set_xticks`.

**Salida esperada:** cuatro cajas; algunos puntos por encima de los bigotes. Un outlier no es automáticamente un error.


In [ ]:
# TODO: crea fig_distribucion y axs_distribucion; calcula grupos y nombres.
# axs_distribucion[0].hist(...); axs_distribucion[1].boxplot(grupos).


In [ ]:
assert len(grupos) == 4
assert len(axs_distribucion) == 2
assert len(axs_distribucion[0].patches) >= 1
assert len(axs_distribucion[1].lines) >= 1
print('Situación 3: comprobación superada')

## Situación 4 — Experiencia y salario

**Pregunta:** ¿la experiencia se asocia con un salario mayor?

**Tarea:** crea `fig_relacion, ax_relacion = plt.subplots(...)`. Dibuja una dispersión de `experiencia` frente a `salario`, con `alpha` y color según la experiencia; añade una colorbar. Calcula `maximo = empleados.loc[empleados['salario'].idxmax()]`, anótalo con `annotate` y añade una línea horizontal con la mediana.

**Salida esperada:** asociación positiva, mucha dispersión y un salario máximo por encima de 10.000 €. La asociación no demuestra causalidad.


In [ ]:
# TODO: crea fig_relacion y ax_relacion; scatter, colorbar, maximo, annotate y mediana.


In [ ]:
assert maximo['salario'] > 10000
assert len(fig_relacion.axes) >= 2
assert len(ax_relacion.collections) >= 1
print('Situación 4: comprobación superada')

## Situación 5 — Panel y recomendación

**Tarea:** crea `fig_panel, axs_panel = plt.subplots(2, 2, ...)` con los cuatro gráficos: líneas, barras, histograma y dispersión. Añade títulos y unidades; no compartas ejes de magnitudes distintas. Anota el máximo y la mediana en la dispersión.

Guarda **antes de** `plt.show()`:

- `salidas_estaticas/panel_personal.png` con `dpi=150` y `bbox_inches='tight'`.
- `salidas_estaticas/panel_personal.svg` con `bbox_inches='tight'`.

Después escribe en una celda Markdown: tres frases (qué muestra cada bloque, qué no permite concluir) y una recomendación para la dirección.


In [ ]:
# TODO: crea fig_panel y axs_panel; cuatro gráficos; guarda PNG y SVG.
# fig_panel.savefig(salida / 'panel_personal.png', dpi=150, bbox_inches='tight')
# fig_panel.savefig(salida / 'panel_personal.svg', bbox_inches='tight')


In [ ]:
assert len(fig_panel.axes) >= 4
assert (salida / 'panel_personal.png').exists()
assert (salida / 'panel_personal.svg').exists()
print('Situación 5: comprobación superada')

## Interpretación

Escribe aquí, en tres frases:

1. Qué muestra cada bloque del panel.
2. Qué **no** permite concluir (por ejemplo, causalidad o generalización).
3. Una recomendación para la dirección, apoyada en una evidencia concreta.

**Respuesta:**

_Por completar._


## Reto opcional

Si terminas antes:

- Compara la media y la mediana **sin los salarios extremos** y explica qué cambia.
- Busca un gráfico engañoso (ejes truncados, tarta con demasiadas categorías, línea para categorías) y describe cómo lo corregirías.
- Añade al panel una comparación de `horas_semana` por departamento.

No es obligatorio para la entrega base.


## Ticket de salida

Responde en dos o tres líneas cada pregunta:

1. ¿Qué gráfico usaste para cada situación y por qué?
2. ¿Qué diferencia hay entre barras e histograma?
3. ¿Qué le hacen los valores extremos a la media? ¿Y a la mediana?
4. ¿Puedes afirmar que la experiencia **causa** un salario mayor? ¿Qué puedes decir?
5. ¿Qué diferencia hay entre `fig` y `ax`?
6. ¿Qué recomendación darías y con qué limitación?

**Respuestas:**

_Por completar._


## Registro de entrega

| Apartado | Completo / en curso / pendiente | Qué he probado o qué falta |
|---|---|---|
| Preparación del entorno | | |
| Situación 1: líneas | | |
| Situación 2: barras e histograma | | |
| Situación 3: cajas y outliers | | |
| Situación 4: dispersión | | |
| Situación 5: panel y exportación | | |
| Interpretación y recomendación | | |

**Ayuda utilizada (documentación, pareja, IA):** indica para qué la usaste y qué comprobaste tú; si no usaste ayuda, escribe «Ninguna».

**Antes de entregar:** reinicia el kernel y ejecuta tus celdas en orden; conserva las salidas. Comprueba la estructura del ZIP en [caso-informe-personal.md](caso-informe-personal.md). Si queda algo pendiente, regístralo y no lo marques como completo.
